# DeepSORT Final Project — Colab

**Runtime:** GPU (T4). **Данные:** `MyDrive/Videos-CV/`

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/danvlak-batya/deep-sort-project.git"
PROJECT_DIR = "/content/deep-sort-project"

import os
os.chdir("/content")

if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)

!python tools/install_colab_deps.py

import ultralytics, timm, torch
print("Проверка OK — ultralytics, timm, torch", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Videos-CV"
import os
assert os.path.isdir(DATA_ROOT)
print(os.listdir(DATA_ROOT))

In [ ]:
from utils.mot_paths import find_sequence_dir, list_image_filenames

SEQUENCE_FOLDER = "Kitti-17"
SEQUENCE = "KITTI-17"
DETECTOR = "yolov8n"
REID = "osnet_x0_25"

SEQ_DIR = find_sequence_dir(DATA_ROOT, SEQUENCE_FOLDER)
print("Кадров:", len(list_image_filenames(SEQ_DIR)))

In [ ]:
import os
os.makedirs("results/demo", exist_ok=True)

!python run_tracker.py \
  --sequence_dir "{SEQ_DIR}" \
  --config configs/default.yaml \
  --detector {DETECTOR} \
  --reid {REID} \
  --output_file results/demo/{SEQUENCE}.txt

p = f"results/demo/{SEQUENCE}.txt"
print("OK:", os.path.exists(p), "size:", os.path.getsize(p) if os.path.exists(p) else 0)

In [ ]:
!python tools/generate_overlays.py \
  --mot_dir "{DATA_ROOT}" \
  --results_dir results/demo \
  --output_dir results/overlays \
  --sequence {SEQUENCE}

from IPython.display import Video, display
v = f"results/overlays/{SEQUENCE}_overlay.mp4"
display(Video(v, embed=True)) if os.path.exists(v) else print("Нет видео")

In [ ]:
RUN_FULL = False
if RUN_FULL:
    !pip install -q trackeval
    !python eval/run_benchmark.py --mot_root "{DATA_ROOT}" --output_dir results/modern
    !python eval/run_mot.py --mot_dir "{DATA_ROOT}" --output_dir results/modern --skip_tracking